# Chapter 3 — Coding attention mechanisms

This notebook develops attention in two stages. First, **simplified attention** uses the input embeddings directly to expose the
core operations: dot-product scores, softmax weights, and weighted context vectors. Next, **self-attention** introduces trainable
query, key, and value projections.

## Learning goals

By the end of this chapter, you should be able to:

- interpret token embedding, attention-score, and context-vector shapes;
- implement simplified attention for one query and for a complete sequence;
- distinguish simplified attention from trainable self-attention; and
- explain the roles of queries, keys, and values in the QKV computation.

In [ ]:
#| default_exp attention

## 3.1 Simplified attention: representing the input sequence

The sentence “Your journey starts with one step” contains six tokens. Each token is represented by a three-dimensional embedding, so
`inputs` has shape `(6, 3)`:

- rows correspond to token positions;
- columns correspond to embedding features.

In this simplified attention mechanism, the same input vectors act as queries, keys, and values. There are no learned projection
matrices yet; that additional trainable step begins in section 3.7.

In [1]:
import torch

# Each row is a three-dimensional embedding for one token in the sentence.
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

C:\Users\giloz\dev\build-llms-from-scratch-companion\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 3.2 Computing attention scores for one query

To determine which tokens are relevant to “journey,” use its embedding as the **query**. Compute one dot product between that query
and every input embedding.

The result `attn_scores_2` has shape `(6,)`: one raw alignment score for each token. A larger score means stronger vector alignment,
but these scores are not yet probabilities and do not need to sum to one.

In [2]:
# Use the second token, "journey," as the query for this example.
query = inputs[1]
print(query)

# torch.empty allocates storage without initializing values; every slot is
# overwritten in the loop before the scores are used.
attn_scores_2 = torch.empty(inputs.shape[0])
print(attn_scores_2.shape)

# A dot product measures the alignment between the query and each input token.
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.5500, 0.8700, 0.6600])
torch.Size([6])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Dot product worked example

For vectors $a$ and $b$, their dot product multiplies corresponding components and sums the results:

$$
a \cdot b = \sum_j a_j b_j
$$

The next cell computes the score between the first token and the query manually, then compares it with `torch.dot`. This confirms
what the vectorized PyTorch operation does.

In [3]:
# Expand one dot product into scalar multiplications and additions.
res = 0.0
for idx, _ in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

# Both calculations should produce the same attention score.
print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


## 3.3 Normalizing attention scores

Attention weights express each token's relative contribution. A simple first approach divides every score by the sum of all scores.
This produces weights that sum to one for this positive-valued example.

This normalization is useful for intuition, but it is not the standard attention rule. It can behave poorly when scores are negative
or their sum is near zero.

In [4]:
# Divide each score by the total to obtain nonnegative weights summing to one.
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


### Normalizing with softmax

Softmax exponentiates each score and divides by the sum of all exponentiated scores:

$$
\mathrm{softmax}(s)_i =
\frac{\exp(s_i)}{\displaystyle\sum_{j=1}^{n} \exp(s_j)}
$$

Here, $s_i$ is the score for token $i$, and $n$ is the number of tokens. The output is positive, sums to one, and gives relatively
larger weights to stronger scores. The implementation below is educational; production code should use `torch.softmax`, which is
optimized and numerically more stable.

In [5]:
def softmax_naive(x: torch.Tensor) -> torch.Tensor:
    """Convert a one-dimensional score tensor into normalized probabilities."""
    return torch.exp(x) / torch.exp(x).sum(dim=0)


# Exponentiation emphasizes larger alignment scores before normalization.
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


### Using PyTorch's stable softmax

The naive implementation repeats `torch.exp(x)` and can overflow for large scores. `torch.softmax` performs the same normalization
with a more numerically stable and optimized implementation.

Here, `dim=0` means “normalize across the token dimension.” Because `attn_scores_2` contains one score per token, the resulting six
weights describe how strongly this query attends to each position.

In [6]:
# Normalize across the six token scores with PyTorch's stable implementation.
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


## 3.4 Computing the context vector

Attention weights become useful when they are applied to the input embeddings. Multiply each input vector by its scalar attention
weight and add the weighted vectors together.

In compact notation, the context vector for query 2 is:

$$
\mathbf{z}^{(2)} = \sum_i \alpha_i^{(2)} \mathbf{x}^{(i)}
$$

Each $\alpha_i^{(2)}$ is a scalar weight and each $\mathbf{x}^{(i)}$ is a three-dimensional input vector. Their weighted sum is
therefore another three-dimensional vector. Tokens receiving larger weights contribute more strongly to the result.

In [7]:
# Reuse "journey" as the query whose context vector we are constructing.
query = inputs[1]
# Start with a zero vector having the same embedding dimensions as the query.
context_vec_2 = torch.zeros(query.shape)

# Scale each input vector by its attention weight, then add the results.
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## 3.5 Extending attention to every query token

The earlier calculation produced one score vector and one context vector for “journey.” Self-attention requires the same operation
for every token in the sequence.

A square attention-score matrix stores all pairwise comparisons. Row `i` contains the scores produced when token `i` acts as the
query; column `j` records its alignment with token `j`. With six tokens, this matrix has shape `(6, 6)`.

In [8]:
# Allocate one score for every query-token and key-token pair.
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        # Row i is the query; column j is the token being scored.
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### Vectorizing pairwise dot products

The nested loops clarify the indexing, but matrix multiplication performs the same work more directly. `inputs` has shape `(6, 3)`
and `inputs.T` has shape `(3, 6)`, so their product has shape `(6, 6)`.

Each output entry is a dot product between one input row and another. The matrix is symmetric here because the same vectors serve on
both sides of the dot product.

In [9]:
# Matrix multiplication computes all pairwise dot products at once.
print(inputs.shape)
attn_scores = inputs @ inputs.T
print(attn_scores)

torch.Size([6, 3])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### Applying softmax row by row

Every row belongs to a different query and must become its own attention distribution. `dim=-1` selects the last dimension—the six
columns in each row—so PyTorch normalizes each query's scores independently.

The resulting matrix keeps shape `(6, 6)`, but its entries are now positive attention weights rather than unrestricted scores.

In [10]:
# Normalize each query row independently across all key positions.
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


### Checking the attention distributions

A valid attention distribution sums to one. The manually rounded values for row 2 total approximately one, while
`attn_weights.sum(dim=-1)` checks all six rows using the full-precision tensor values.

In [11]:
# Verify one displayed row manually, then check every row with PyTorch.
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 3.6 Computing all context vectors

Each attention row contains the coefficients for one weighted sum of the input embeddings. Matrix multiplication performs all six
weighted sums simultaneously:

- attention weights: `(6, 6)`;
- input embeddings: `(6, 3)`;
- context vectors: `(6, 3)`.

Row `i` of the result is the context vector for query token `i`.

In [12]:
# Multiply every attention row by the input matrix to obtain all contexts.
all_context_vecs = attn_weights @ inputs
print(attn_weights.shape)
print(inputs.shape)
print(all_context_vecs)

torch.Size([6, 6])
torch.Size([6, 3])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### Verifying the vectorized result

The second row of `all_context_vecs` corresponds to “journey,” whose context vector was previously calculated with an explicit loop.
Comparing the two confirms that the matrix operation reproduces the single-query calculation.

In [13]:
# Row 1 of all_context_vecs matches this earlier result for the second token.
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


## Simplified attention checkpoint

The simplified attention calculation is now complete for the full sequence:

1. `inputs @ inputs.T` uses the input embeddings directly to compute pairwise scores;
2. `softmax(..., dim=-1)` normalizes every query row independently;
3. each row becomes one token's attention distribution; and
4. `attn_weights @ inputs` produces one context vector for every token.

This version captures the essential attention pattern, but it cannot learn different representations for matching tokens and
carrying information. Trainable self-attention solves that limitation with separate query, key, and value projections.

## 3.7 Self-attention with trainable QKV projections

Simplified attention reused each input embedding for all three roles. In trainable **self-attention**, every input vector is projected
into three different representations:

- a **query** asks what information the current token needs;
- a **key** describes what information a token can be matched on;
- a **value** contains the information contributed to a context vector.

For an input matrix `x`, the projections are computed with learned weight matrices:

- `queries = x @ W_query`
- `keys = x @ W_key`
- `values = x @ W_value`

Attention scores then compare queries with keys, while the normalized weights combine the values. The following cells will implement
these QKV operations first for one token and then for the complete sequence.

### Choosing input and projection dimensions

Begin with one token so each shape remains easy to follow. `x_2` is the three-dimensional embedding for “journey,” so `d_in=3`.
The projection width `d_out=2` means each query, key, and value will contain two learned features.

In a full model, these dimensions are architectural hyperparameters. The query and key widths must match because their dot product
produces attention scores; using the same value width is convenient here but not required by the score calculation.

In [14]:
# Focus on the second token while developing the QKV calculation.
x_2 = inputs[1]
# d_in is the original token-embedding width.
d_in = inputs.shape[1]  # d_model
# Use a smaller shared output width for queries, keys, and values.
d_out = 2  # d_k, d_q, d_v

### Initializing QKV projection matrices

Self-attention learns three independent matrices, each with shape `(d_in, d_out)`. Although their shapes match, they have different
parameters and therefore learn different transformations.

`requires_grad=False` freezes them only for this hand-worked demonstration. A trainable attention module would leave gradient
tracking enabled so optimization can update all three matrices.

In [15]:
# Fix the seed so the demonstration uses reproducible projection weights.
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

### Projecting one token into query, key, and value vectors

Multiplying `x_2` with each weight matrix changes its representation from shape `(d_in,)` to `(d_out,)`.

For `query_2 = x_2 @ W_query`, the shared `d_in` dimension is contracted by matrix multiplication:

```text
(3,) @ (3, 2) -> (2,)
```

No broadcasting is needed. The same rule creates `key_2` and `value_2`, but with independently initialized weights.

In [16]:
# Confirm the inner dimensions agree before matrix multiplication.
print(x_2.shape)
print(W_query.shape)

# (d_in,) @ (d_in, d_out) contracts d_in and returns shape (d_out,).
# This is vector-matrix multiplication; it does not require broadcasting.
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

torch.Size([3])
torch.Size([3, 2])
tensor([0.4306, 1.4551])


### Projecting the complete sequence

Matrix multiplication can project all tokens simultaneously. `inputs` has shape `(6, 3)` and each projection matrix has shape
`(3, 2)`, producing `keys` and `values` with shape `(6, 2)`.

Each output row belongs to the token at the same sequence position as the corresponding input row.

In [17]:
# Project all six token embeddings into key and value spaces at once.
print(inputs.shape)
print(W_key.shape)
print(W_value.shape)
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

torch.Size([6, 3])
torch.Size([3, 2])
torch.Size([3, 2])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


### Computing one query-key score

A self-attention score is the dot product of a query with a key. `attn_score_22` compares the second token's query with the second
token's key, so it measures how strongly “journey” attends to itself before scaling and normalization.

In [18]:
# Compare the second token's query with its own key vector.
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


### Scoring one query against every key

To discover how the second token relates to the full sequence, multiply its query by the transposed key matrix:

```text
query_2: (2,)
keys.T:  (2, 6)
result:  (6,)
```

The six resulting values are raw attention scores—one for each key position. They still need scaling and softmax before they can be
used as weights for combining value vectors.

In [19]:
# Compare the second query with every key using one matrix operation.
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


### Raw QKV score checkpoint

For the second token, the notebook has now:

1. projected its input embedding into separate query, key, and value vectors;
2. projected the full sequence into key and value matrices;
3. computed one self-attention score with a query-key dot product; and
4. vectorized that comparison across every key in the sequence.

These are still raw dot-product scores. Scaled dot-product attention divides them by the square root of the key dimension before
applying softmax.

### Scaled dot-product attention

Vaswani et al. introduce **scaled dot-product attention** in *Attention Is All You Need*:

$$
\mathrm{Attention}(Q, K, V) =
\mathrm{softmax}\left(\frac{QK^{\mathsf{T}}}{\sqrt{d_k}}\right)V
$$

Here, `Q`, `K`, and `V` are the query, key, and value matrices, and $d_k$ is the width of each key vector.

As $d_k$ grows, unscaled query-key dot products can become large. Large values push softmax toward extremely peaked outputs, where gradients may become very small. Dividing by the square root of $d_k$ keeps score magnitudes in a more useful range.

This cell has one query rather than a complete `Q` matrix, so it applies the same rule to `attn_scores_2` before normalizing across the six key positions.

In [20]:
# d_k is the width of each key and query vector.
d_k = keys.shape[-1]
# Scale by sqrt(d_k) before softmax to control the magnitude of dot products.
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


### Scaled-attention checkpoint

The second query's raw scores have now been divided by `sqrt(d_k)` and normalized with softmax. The resulting six attention weights
are positive and describe how strongly the query attends to each key position.

The remaining step for this query is to use these weights in a weighted sum of the projected `values`, producing its self-attention
context vector.

### Combining projected values

The attention weights describe how much each key position contributes to the second query. Multiplying this six-element weight
vector by `values` of shape `(6, d_v)` produces one context vector of shape `(d_v,)`.

Unlike simplified attention, this weighted sum combines **projected value vectors**, not the original input embeddings.

In [21]:
# Use the normalized weights to combine the projected value vectors.
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


## 3.8 Packaging self-attention in a PyTorch module

`SelfAttention_v1` combines the QKV projections, scaled scores, softmax weights, and value aggregation into one reusable layer. It is
a pedagogical **single attention head**, not yet a complete Transformer attention block.

Given input shape `(N, d_in)`, the layer returns `(N, d_out)` because its value vectors have width `d_out`. A residual Transformer
block needs an output shaped `(N, d_model)`. That contract is restored later by either:

- choosing `d_out == d_model` for a single head; or
- concatenating multiple heads and applying an output projection back to `d_model`.

This version also uses `.T`, so it expects an unbatched two-dimensional input. Batched attention will use
`transpose(-2, -1)` instead.

In [22]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    """Compute single-head self-attention with explicit weight parameters."""

    def __init__(self, d_in: int, d_out: int) -> None:
        """Initialize query, key, and value projections."""
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return one projected context vector per input token."""
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        # Each row scores one query against every key position.
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1,
        )
        # The output width is d_out because values has width d_out.
        context_vec = attn_weights @ values
        return context_vec

### Running `SelfAttention_v1`

With six input tokens and `d_out=2`, the module returns shape `(6, 2)`: one two-dimensional context vector per token. The second row
matches the context vector calculated manually because the random seed recreates the same QKV parameters.

In [23]:
# Reproduce the hand-worked QKV example with the module implementation.
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


## 3.9 Replacing explicit parameters with `nn.Linear`

`SelfAttention_v2` uses `nn.Linear` for the three projections. A linear layer provides standard PyTorch initialization, optional
biases, device handling, serialization, and integration with model tooling.

For an input `x`, `nn.Linear(d_in, d_out)` computes `x @ weight.T + bias`. This explains why its stored weight shape is the transpose
of the explicit matrices used in version 1.

In [24]:
class SelfAttention_v2(nn.Module):
    """Compute single-head self-attention with nn.Linear projections."""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        qkv_bias: bool = False,
    ) -> None:
        """Initialize optional-bias query, key, and value layers."""
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return one projected context vector per input token."""
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # This pedagogical form expects an unbatched (tokens, d_in) tensor.
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1,
        )
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
%%sql


### Running `SelfAttention_v2`

The output shape remains `(6, 2)`, but the values differ from version 1 because `nn.Linear` uses a different initialization scheme and
the example uses a different random seed. Different initial outputs do not imply different attention logic.

In [25]:
# A different seed and initialization scheme produce different initial outputs.
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


### Self-attention module checkpoint

Both implementations now perform the same conceptual pipeline: project QKV, calculate scaled query-key scores, normalize each query
row, and combine value vectors.

They return projected context vectors with width `d_out`. A complete multi-head Transformer attention layer will concatenate head
outputs and use an output projection so the residual stream returns to `d_model`.

## Exercise 3.1 — Comparing `SelfAttention_v1` and `SelfAttention_v2`

The two classes implement the same attention equations but initialize and store their projection weights differently.
`SelfAttention_v1` uses matrices with shape `(d_in, d_out)`, while `nn.Linear` stores its weight with shape `(d_out, d_in)` and
internally applies the transpose.

Transfer the query, key, and value weights from a `SelfAttention_v2` instance to a `SelfAttention_v1` instance. Account for this
transpose when copying each matrix. After the transfer, both modules should produce matching outputs for the same input.

This exercise separates differences caused by parameter initialization from differences in the attention computation itself.

### Solution

Create fresh, exercise-specific instances so the earlier `sa_v1` and `sa_v2` examples remain unchanged. `nn.Linear.weight` stores
shape `(d_out, d_in)`, whereas version 1 expects `(d_in, d_out)`, so each source matrix must be transposed during the copy.

The copy runs inside `torch.no_grad()` because it initializes parameters rather than defining a differentiable model operation.
`torch.random.fork_rng()` also restores the previous random-number-generator state afterward, preventing this exercise from changing
random initialization in later cells.

In [26]:
# Keep all exercise objects isolated behind an exo_3_1_ prefix.
with torch.random.fork_rng():
    torch.manual_seed(301)
    exo_3_1_attention_v1 = SelfAttention_v1(d_in, d_out)
    exo_3_1_attention_v2 = SelfAttention_v2(d_in, d_out)

# nn.Linear stores (d_out, d_in), so transpose into v1's (d_in, d_out).
with torch.no_grad():
    exo_3_1_attention_v1.W_query.copy_(exo_3_1_attention_v2.W_query.weight.T)
    exo_3_1_attention_v1.W_key.copy_(exo_3_1_attention_v2.W_key.weight.T)
    exo_3_1_attention_v1.W_value.copy_(exo_3_1_attention_v2.W_value.weight.T)

exo_3_1_output_v1 = exo_3_1_attention_v1(inputs)
exo_3_1_output_v2 = exo_3_1_attention_v2(inputs)
exo_3_1_outputs_match = torch.allclose(
    exo_3_1_output_v1,
    exo_3_1_output_v2,
)
exo_3_1_max_difference = (exo_3_1_output_v1 - exo_3_1_output_v2).abs().max()

print("Outputs match:", exo_3_1_outputs_match)
print("Maximum absolute difference:", exo_3_1_max_difference)

Outputs match: True
Maximum absolute difference: tensor(0., grad_fn=<MaxBackward1>)


## 3.10 Causal attention

A decoder-only language model predicts tokens from left to right. During training, the complete sequence is available in one tensor,
but each position must be prevented from reading tokens that occur later in that sequence. Otherwise, the model could “cheat” by
using the answers it is supposed to predict.

**Causal attention** enforces this constraint with a lower-triangular mask. Query row `i` may attend only to key columns `j <= i`:
the current token and all earlier tokens.

### Starting from unmasked attention

First compute the ordinary QKV attention weights. Every row currently assigns nonzero probability to all six key positions,
including future tokens. This matrix provides the baseline that the causal mask will restrict.

In [27]:
# Recompute unmasked self-attention weights with the version 2 projections.
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
print(attn_scores)
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1] ** 0.5,
    dim=-1,
)
print(attn_weights)

tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)
tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Constructing a lower-triangular mask

`torch.tril` keeps the diagonal and everything below it:

- row 0 keeps only column 0;
- row 1 keeps columns 0 and 1;
- the final row keeps every column.

Ones mark allowed query-key connections, while zeros mark future positions that must be hidden.

In [28]:
# A lower-triangular mask keeps the current and all previous positions.
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


### Removing attention to future positions

Element-wise multiplication sets every disallowed future weight to zero. The allowed values retain their old magnitudes, so the rows
no longer necessarily sum to one after masking.

In [29]:
# Zero every attention weight that points to a future token.
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


### Renormalizing the masked rows

Divide each row by its remaining sum to recover a valid probability distribution. `keepdim=True` preserves shape `(N, 1)`, allowing
PyTorch to broadcast one row sum across the corresponding `N` columns.

This post-softmax masking is useful for understanding the mechanics. The standard implementation instead fills forbidden **scores**
with negative infinity before softmax, which masks and normalizes in one operation.

In [30]:
# Masking after softmax reduces row totals, so normalize each row again.
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


### Post-softmax masking checkpoint

The first causal implementation multiplied already-normalized weights by a lower-triangular mask and then renormalized each row. It
makes both steps visible, but practical attention implementations usually mask the raw scores before softmax.

The next approach produces the same causal distributions without an explicit renormalization step.

### Masking raw scores with negative infinity

Create an upper-triangular mask above the main diagonal. These entries represent future key positions: column `j` is forbidden when
`j > i` for query row `i`.

`masked_fill` replaces each forbidden raw score with negative infinity. Allowed scores, including the diagonal, remain unchanged.
This reverses the earlier mask convention: here `True` means “hide this position.”

In [31]:
# Mark strictly future positions above the main diagonal.
mask = torch.triu(
    torch.ones(context_length, context_length),
    diagonal=1,
)
# Replace forbidden raw scores with negative infinity before softmax.
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


### Letting softmax enforce the causal constraint

Exponentiating negative infinity gives zero, so masked positions automatically receive exactly zero probability. Softmax normalizes
only the finite scores that remain in each row, eliminating the separate renormalization step.

`dim=1` selects the columns of this two-dimensional `(queries, keys)` matrix. For a future batched implementation, `dim=-1` expresses
the same intent more generally by always selecting the final key dimension.

In [32]:
# Scale and normalize each query row; softmax maps -inf entries to zero.
attn_weights = torch.softmax(
    masked / keys.shape[-1] ** 0.5,
    dim=1,
)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Pre-softmax causal masking checkpoint

The causal attention pipeline now follows the standard order:

1. compute scaled query-key scores;
2. replace scores for future positions with negative infinity;
3. apply softmax across each query's key positions; and
4. obtain normalized rows whose future-facing weights are exactly zero.

This mask can be registered once inside an attention module and sliced to the active sequence length during each forward pass.

## 3.11 Applying dropout to attention weights

Dropout regularizes attention by randomly removing query-key connections during training. Surviving values are scaled by
`1 / (1 - p)` so their expected magnitude remains unchanged. It is disabled when the module is in evaluation mode.

### Inspecting dropout on ones

A matrix of ones makes dropout visible: with `p=0.5`, dropped entries become zero and retained entries become two. A fixed seed makes
the demonstration reproducible.

In [33]:
# Fix the seed so the dropout pattern is reproducible.
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


### Dropping attention connections

Applying dropout after softmax removes selected attention links. Individual sampled rows need not sum to one; dropout scaling
preserves the expected activation across many samples.

In [34]:
# Apply the same dropout rate to the causal attention weights.
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


## 3.12 Packaging batched causal attention

`CausalAttention` combines QKV projections, scaled scores, a causal mask, softmax, dropout, and value aggregation. It maps
`(batch, tokens, d_in)` to `(batch, tokens, d_out)`. The mask is a registered buffer, not a trainable parameter.

### Creating a batch dimension

Stacking two copies changes shape `(6, 3)` into `(2, 6, 3)`: two sequences, six token positions, and three input features.

In [35]:
# Duplicate the sequence to introduce an explicit batch dimension.
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [36]:
class CausalAttention(nn.Module):
    """Compute batched, single-head causal self-attention."""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float,
        qkv_bias: bool = False,
    ) -> None:
        """Initialize QKV projections and a fixed causal mask."""
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            # mask: (context_length, context_length)
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map `(batch, tokens, d_in)` to `(batch, tokens, d_out)`."""
        # x: (batch, num_tokens, d_in)
        # d_in = d_model (d_embed)
        b, num_tokens, d_in = x.shape

        # keys, queries, values: (batch, num_tokens, d_out)
        # d_out = d_k = d_q = d_v
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # keys.transpose(1, 2): (batch, d_out, num_tokens)
        # attn_scores: (batch, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(1, 2)

        # mask slice: (num_tokens, num_tokens), broadcast across batch
        # attn_scores remains: (batch, num_tokens, num_tokens)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf,
        )

        # attn_weights: (batch, num_tokens, num_tokens)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1,
        )
        attn_weights = self.dropout(attn_weights)

        # (batch, num_tokens, num_tokens) @ (batch, num_tokens, d_out)
        # context_vec: (batch, num_tokens, d_out)
        context_vec = attn_weights @ values
        return context_vec

### Running batched causal attention

Input `(2, 6, 3)` with `d_out=2` becomes `(2, 6, 2)`. Batch and token dimensions are preserved while each token receives a
two-dimensional causal context vector.

In [37]:
# Run batched causal attention without dropout for easier inspection.
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


## 3.13 Building a simple multi-head attention wrapper

Independent heads can learn different token relationships. This wrapper runs several `CausalAttention` modules and concatenates
their outputs. The final width is `num_heads * d_out`; a complete Transformer later projects this width back to `d_model`.

In [38]:
class MultiHeadAttentionWrapper(nn.Module):
    """Run independent causal-attention heads and concatenate their outputs."""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float,
        num_heads: int,
        qkv_bias: bool = False,
    ) -> None:
        """Create `num_heads` independent causal-attention modules."""
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Concatenate head outputs along the embedding dimension."""
        # Each head: (batch, tokens, d_out)
        # Result: (batch, tokens, num_heads * d_out)
        return torch.cat([head(x) for head in self.heads], dim=-1)

### Concatenating two heads

Each head returns `(2, 6, 2)`. Concatenating two outputs along the last dimension produces `(2, 6, 4)`.

In [39]:
# Two heads of width two produce a concatenated width of four.
torch.manual_seed(123)
context_length = batch.shape[1]  # Number of tokens in each sequence.
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


## Exercise 3.2 — Returning two-dimensional context vectors

Keep `num_heads=2` while making the concatenated output width two. Since the width is `num_heads * d_out`, each head must use
`d_out=1`. The solution uses exercise-prefixed variables to preserve earlier examples.

### Solution

Reduce each of the two heads to width one. Concatenation then produces `2 * 1 = 2` features per token without changing batch or
sequence dimensions.

In [40]:
# Two heads of width one concatenate into two-dimensional vectors.
exo_3_2_d_in, exo_3_2_d_out = 3, 1
exo_3_2_mha = MultiHeadAttentionWrapper(
    exo_3_2_d_in, exo_3_2_d_out, context_length, 0.0, num_heads=2
)
exo_3_2_context_vecs = exo_3_2_mha(batch)
print(exo_3_2_context_vecs)
print("context_vecs.shape:", exo_3_2_context_vecs.shape)

tensor([[[0.0189, 0.2729],
         [0.2181, 0.3037],
         [0.2804, 0.3125],
         [0.2830, 0.2793],
         [0.2476, 0.2541],
         [0.2748, 0.2513]],

        [[0.0189, 0.2729],
         [0.2181, 0.3037],
         [0.2804, 0.3125],
         [0.2830, 0.2793],
         [0.2476, 0.2541],
         [0.2748, 0.2513]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


### Multi-head wrapper checkpoint

The notebook now supports batched causal attention, attention dropout, and multiple independent heads. Concatenated width follows
`num_heads * d_out`.

A production implementation computes heads with combined projections, reshapes rather than looping over modules, and applies an
output projection back to `d_model` for the residual connection.

## 3.14 Computing multi-head attention efficiently

The wrapper above creates one complete `CausalAttention` module per head. A practical implementation instead computes all heads with
one query projection, one key projection, and one value projection, then reshapes the resulting feature dimension into separate
heads.

This removes the Python-level loop from the forward pass and lets PyTorch process all heads with batched tensor operations. The
attention calculation is unchanged; only its organization is more efficient.

This notebook remains the source of truth for the reusable implementation. The `#| export` directive on the class cell tells nbdev
to generate `build_llms_from_scratch_companion/attention.py`; other notebooks import the generated module instead of executing this
notebook.

### Shape roadmap

Let `B` be batch size, `T` sequence length, `H` number of heads, and `D_h = d_out / H` the width of one head.

The forward pass transforms tensors as follows:

```text
input                       (B, T, d_in)
combined Q, K, V            (B, T, d_out)
split into heads            (B, T, H, D_h)
transpose for attention     (B, H, T, D_h)
attention scores/weights    (B, H, T, T)
per-head context vectors    (B, H, T, D_h)
merge heads                 (B, T, d_out)
output projection           (B, T, d_out)
```

The assertion `d_out % num_heads == 0` guarantees that every head receives the same integer width.

In [ ]:
#| export
import torch
from torch import nn

In [41]:
#| export
class MultiHeadAttention(nn.Module):
    """Compute batched causal self-attention with multiple parallel heads."""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float,
        num_heads: int,
        qkv_bias: bool = False,
    ) -> None:
        """Initialize combined QKV projections, causal mask, and output layer."""
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        # d_out is the total width after all heads are concatenated.
        self.d_out = d_out
        self.num_heads = num_heads
        # Each head receives an equal slice of the projected features.
        self.head_dim = d_out // num_heads
        # Each projection computes every head in one linear operation.
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # Mix information across concatenated heads without changing width.
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            # mask: (context_length, context_length)
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map `(batch, tokens, d_in)` to `(batch, tokens, d_out)`."""
        # x: (batch, num_tokens, d_in)
        b, num_tokens, d_in = x.shape

        # Combined projections: (batch, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split d_out into heads:
        # (batch, num_tokens, d_out)
        # -> (batch, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Move heads before tokens for parallel attention:
        # (batch, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # keys.transpose(2, 3): (batch, num_heads, head_dim, num_tokens)
        # attn_scores: (batch, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # mask_bool: (num_tokens, num_tokens), broadcast over batch and heads
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # attn_scores remains: (batch, num_heads, num_tokens, num_tokens)
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # attn_weights: (batch, num_heads, num_tokens, num_tokens)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1,
        )
        attn_weights = self.dropout(attn_weights)

        # Weighted values: (batch, num_heads, num_tokens, head_dim)
        # After transpose: (batch, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Merge heads: (batch, num_tokens, num_heads * head_dim)
        # Since num_heads * head_dim == d_out: (batch, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        # out_proj preserves: (batch, num_tokens, d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

### Why `.contiguous()` is needed before merging heads

A PyTorch tensor combines underlying storage with metadata describing its shape and **strides**. Strides tell PyTorch how far to move
through storage when an index changes.

`transpose(1, 2)` normally returns a view without copying values. It swaps dimensions by changing strides, so the logical tensor has
shape `(B, T, H, D_h)` even though its values are not laid out in that order in memory. Such a view is usually **non-contiguous**.

```python
context_vec = (attn_weights @ values).transpose(1, 2)
context_vec = context_vec.contiguous().view(B, T, d_out)
```

The two operations have distinct roles:

1. `.contiguous()` returns the same logical values in standard row-major memory order. It returns the original tensor when the layout
   is already contiguous; otherwise, it allocates storage and copies the values.
2. `.view(B, T, d_out)` then reinterprets that contiguous storage with `H` and `D_h` merged into one dimension.

Without `.contiguous()`, `view` may fail because the requested merged dimension is incompatible with the transposed strides. Using
`.reshape(...)` is an alternative that returns a view when possible and performs a copy when necessary.

This operation does not change the attention values or perform learned computation. It only makes their memory layout suitable for
merging the head dimensions. Afterward, `out_proj` mixes features across heads while preserving shape `(B, T, d_out)`. A residual
Transformer block normally chooses `d_out == d_model`.

### Efficient multi-head attention checkpoint

The optimized module now performs QKV projection, head splitting, causal scaled dot-product attention, head merging, and output
projection entirely with batched tensor operations.

Its contract is `(B, T, d_in) -> (B, T, d_out)`. When `d_in` and `d_out` both equal `d_model`, its result has the shape required by a
Transformer residual connection.

illustrate batched matrix mult

In [42]:
a = torch.tensor(
    [
        [
            [
                [0.2745, 0.6584, 0.2775, 0.8573],
                [0.8993, 0.0390, 0.9268, 0.7388],
                [0.7179, 0.7058, 0.9156, 0.4340],
            ],
            [
                [0.0772, 0.3565, 0.1479, 0.5331],
                [0.4066, 0.2318, 0.4545, 0.9737],
                [0.4606, 0.5159, 0.4220, 0.5786],
            ],
        ]
    ]
)
print(a.shape)

torch.Size([1, 2, 3, 4])


In [43]:
print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [44]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [45]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Exercise 3.3 Initializing GPT-2 size attention modules

Using the MultiHeadAttention class, initialize a multi-head attention module that has the same number of attention heads as the smallest GPT-2 model (12 attention heads). Also ensure that you use the respective input and output embedding sizes similar to GPT-2 (768 dimensions). Note that the smallest GPT-2 model supports a context length of 1,024 tokens.

In [48]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
exo_3_3_d_out = 768
exo_3_3_mha = MultiHeadAttention(d_in, exo_3_3_d_out, context_length, 0.0, num_heads=12)
exo_3_3_context_vecs = exo_3_3_mha(batch)
print(exo_3_3_context_vecs)
print("context_vecs.shape:", exo_3_3_context_vecs.shape)

tensor([[[ 0.0208, -0.1094, -0.1502,  ...,  0.3617,  0.2821,  0.0099],
         [-0.0732, -0.1550, -0.1058,  ...,  0.4179,  0.2185,  0.0626],
         [-0.1013, -0.1662, -0.0936,  ...,  0.4298,  0.1946,  0.0779],
         [-0.1035, -0.1574, -0.0720,  ...,  0.3876,  0.1603,  0.0761],
         [-0.0765, -0.1191, -0.0922,  ...,  0.3362,  0.1465,  0.0587],
         [-0.0913, -0.1358, -0.0698,  ...,  0.3519,  0.1339,  0.0640]],

        [[ 0.0208, -0.1094, -0.1502,  ...,  0.3617,  0.2821,  0.0099],
         [-0.0732, -0.1550, -0.1058,  ...,  0.4179,  0.2185,  0.0626],
         [-0.1013, -0.1662, -0.0936,  ...,  0.4298,  0.1946,  0.0779],
         [-0.1035, -0.1574, -0.0720,  ...,  0.3876,  0.1603,  0.0761],
         [-0.0765, -0.1191, -0.0922,  ...,  0.3362,  0.1465,  0.0587],
         [-0.0913, -0.1358, -0.0698,  ...,  0.3519,  0.1339,  0.0640]]],
       grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 768])
